In [27]:
import numpy as np, math
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar

import matplotlib.pyplot as plt
from scipy.optimize import brentq

In [28]:
# Constants
G=6.674e-11 
mE=5.972e24
mM=7.348e22
d=3.844e8

In [33]:
# System paramters and basic motion


w =math.sqrt(G*(mE+mM)/d**3) # angular velocity of the system

rE= mM*d/(mE+mM) # distance of earth from barycentre
rM= mE*d/(mE+mM) # distance of moon from barycentre


def body_pos(t): 
    cos_comp ,sine_comp = np.cos(w*t), np.sin(w*t)
    return ((-mM*d/(mE+mM)*cos_comp, -mM*d/(mE+mM)*sine_comp), (mE*d/(mE+mM)*cos_comp, mE*d/(mE+mM)*sine_comp))

def body_vel(t):
    cos_comp ,sine_comp = np.cos(w*t), np.sin(w*t)
    return ((mM*d/(mE+mM)*w*sine_comp, -mM*d/(mE+mM)*w*cos_comp), (-mE*d/(mE+mM)*w*sine_comp, mE*d/(mE+mM)*w*cos_comp))

def initial_conditions_planests(t0):                         # gives positon and velocity of each body at the start
    pos = body_pos(t0)
    vel = body_vel(t0)
    return (pos[0][0], pos[0][1], pos[1][0], pos[1][1],
            vel[0][0], vel[0][1], vel[1][0], vel[1][1])

def init(dr,t):                                                 #initial position of satellite dr past the moon at a time t 
    cos_comp ,sine_comp = np.cos(w*t), np.sin(w*t)
    return ((rM+dr)*cos_comp, (rM+dr)*sine_comp, 
            -w*(rM+dr)*sine_comp, w*(rM+dr)*cos_comp)

def func_to_solve(t, s):
    x, y, vx, vy = s             
    (xE, yE),(xM, yM) = body_pos(t)
    dEx,dEy,dMx,dMy=x-xE,y-yE,x-xM,y-yM
    dE3=(dEx*dEx+dEy*dEy)**1.5; dM3=(dMx*dMx+dMy*dMy)**1.5
    return np.array([vx, vy , -G*mE*dEx/dE3 - G*mM*dMx/dM3, -G*mE*dEy/dE3 - G*mM*dMy/dM3])


In [34]:
#Scipy Solver 
def Scipy_solver(func,t_span, s, t_eval):
    sol = solve_ivp(func, t_span, s , t_eval=t_eval, method='RK45', rtol=1e-9, atol=1e-9)
    return sol 

#Scipy Solver with dense output  - FIX COS TS BROKEN
def Scipy_solver_dense(func, T0, T, n, dr):
    s = init(dr, T0)
    t = np.linspace(T0, T, n)
    sol = solve_ivp(func, [T0, T], s,t_eval=t,method='RK45',rtol=1e-9,atol=1e-9,dense_output=True)
    return sol


In [35]:
#Taylor Solver -ADD JERK LATER - THINK ABOUT HOW I STORE THE DATA IM USING AND WHAT TIMESTEP PARAMETRS TO USE
def Taylor_solver(dr , T0, T,n):
    pos = np.array([init(dr, T0)])    
    dt= (T-T0)/n
    times = np.array([T0])
    for i in  range(0,n):
        t=T0+i*dt
        new_X = pos[-1][0] + dt*pos[-1][2] + 0.5*dt*dt*func_to_solve(times[-1],pos[-1])[2]
        new_y = pos[-1][1] + dt*pos[-1][3] + 0.5*dt*dt*func_to_solve(times[-1],pos[-1])[3]
        new_vx = pos[-1][2] + dt*func_to_solve(times[-1],pos[-1])[2]
        new_vy = pos[-1][3] + dt*func_to_solve(times[-1],pos[-1])[3]
        pos = np.append(pos, [[new_X, new_y, new_vx, new_vy]], axis=0)
        times = np.append(times, t+dt)
    return times, pos


In [40]:
#Runga Kutta Solver
def rk_step(f,t,u,h):
    k1 = f(t,u)
    k2 = f(t + h/2, u + h*k1/2)
    k3 = f(t + h/2, u + h*k2/2) 
    k4 = f(t + h, u + h*k3)
    return u + h*(k1 + 2*k2 + 2*k3 + k4)/6

def Runga_Kutta_solver(f , T0,dr, T_end,h):
    pos = np.array([init(dr, T0)])    
    times = np.array([T0])
    t=T0
    while t<T_end:
        s = rk_step(f,t,pos[-1], h)
        pos = np.append(pos, [s], axis=0)
        t += h
        times = np.append(times, t)
    return times, pos
    



In [ ]:
#metrics for how accurate it is 

In [ ]:
#plots for each 